# AED v8 — mais dados reais, classificador hierárquico e teste limpo

Esta versão parte do AED v7, preserva o split global sem vazamento e corrige os dois principais gargalos: pouca diversidade real em `impacto` e `choro_distress`, e excesso de dependência de augmentation.

Mudanças:

- FSD50K por streaming, usando apenas clipes úteis e negativos difíceis;
- Baby Crying Sounds e Donate-a-Cry para ampliar choro real;
- os datasets adicionais entram somente no treino;
- nenhum augmentation offline é usado;
- cabeças hierárquicas para família, vocal, transitório e perigo;
- focal loss com peso maior para impacto e choro;
- mineração de exemplos difíceis em uma segunda etapa;
- seleção por macro-F1 combinado na validação;
- média top-3 por gravação;
- treinamento e exportação em float32 para TFLite nativo.

Critério de aceitação:

- acurácia no teste >= 90%;
- F1 macro no teste >= 90%;
- menor F1 entre as classes >= 80%.

A meta é um critério de aceitação, não uma garantia.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg git
!pip -q install \
    kagglehub \
    librosa==0.11.0 \
    soundfile \
    scikit-learn \
    pandas \
    matplotlib \
    tqdm \
    pyyaml \
    requests \
    tensorflow-hub \
    datasets \
    huggingface_hub

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any

import gc
import io
import json
import math
import random
import re
import shutil
import subprocess
import time
import zipfile

import kagglehub
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import soundfile as sf
import tensorflow as tf
import tensorflow_hub as hub

from datasets import Features, Value, load_dataset
from google.colab import files
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("Ative uma GPU T4 no Colab.")

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

# Corrige a falha de exportação do AED v7.
tf.keras.mixed_precision.set_global_policy("float32")
print("Política numérica:", tf.keras.mixed_precision.global_policy())

## 1. Configuração

In [ ]:
@dataclass(frozen=True)
class Configuracao:
    sr: int = 16000
    duracao: float = 3.0
    passo: float = 0.50

    n_mfcc: int = 40
    n_fft: int = 512
    hop_length: int = 256

    max_arquivos_por_classe: int = 5000
    max_janelas_por_arquivo: int = 3
    max_janelas_suplementares: int = 2

    max_fsd_por_classe: int = 1800
    max_fsd_normal: int = 2200
    max_baby_kaggle: int = 1600
    max_donate_cry: int = 900

    lote: int = 64
    epocas_etapa_1: int = 65
    epocas_etapa_2: int = 28
    paciencia: int = 10
    seed: int = 42

    tentativas_split: int = 350
    max_hard_normal: int = 1500
    max_hard_choro: int = 1100
    max_hard_impacto: int = 1100
    repeticoes_dificeis: int = 3

    meta_accuracy: float = 0.90
    meta_f1_macro: float = 0.90
    meta_f1_classe: float = 0.80
    meta_paridade: float = 0.99

    usar_aob: bool = True
    usar_tau_privacy: bool = True
    usar_tut: bool = True
    usar_reforco_manual: bool = False
    usar_fsd50k: bool = True
    usar_baby_kaggle: bool = True
    usar_donate_cry: bool = True


CFG = Configuracao()

CLASSES = [
    "ambiente_normal",
    "grito_panico",
    "choro_distress",
    "impacto",
    "vidro_quebrando",
    "tiro",
]

ID = {classe: indice for indice, classe in enumerate(CLASSES)}

FAMILIA = {
    "ambiente_normal": 0,
    "grito_panico": 1,
    "choro_distress": 1,
    "impacto": 2,
    "vidro_quebrando": 2,
    "tiro": 2,
}

VOCAL = {
    "grito_panico": 0,
    "choro_distress": 1,
}

TRANSITORIO = {
    "impacto": 0,
    "vidro_quebrando": 1,
    "tiro": 2,
}

RARAS = set(TRANSITORIO)

# AED v8 não cria cópias artificiais offline.
AUGMENTACOES = {classe: 0 for classe in CLASSES}

PESOS_AMOSTRAGEM = {
    "ambiente_normal": 0.20,
    "grito_panico": 0.15,
    "choro_distress": 0.17,
    "impacto": 0.18,
    "vidro_quebrando": 0.15,
    "tiro": 0.15,
}

if not np.isclose(sum(PESOS_AMOSTRAGEM.values()), 1.0):
    raise RuntimeError("Pesos de amostragem inválidos.")

RAIZ = Path("/content/aed_v8")
DADOS = RAIZ / "datasets"
CACHE = RAIZ / "cache"
MODELOS = RAIZ / "modelos"
RESULTADOS = RAIZ / "resultados"
ENTREGA = RAIZ / "entrega"

ESC_DIR = DADOS / "esc50"
TUT_DIR = DADOS / "tut_rare"
AOB_DEV = DADOS / "aob_dev"
AOB_EVAL = DADOS / "aob_eval"
TAU_DIR = DADOS / "tau_privacy"
REFORCO = DADOS / "reforco_manual"
DONATE_DIR = DADOS / "donate_a_cry"

for pasta in (RAIZ, DADOS, CACHE, MODELOS, RESULTADOS, ENTREGA):
    pasta.mkdir(parents=True, exist_ok=True)

random.seed(CFG.seed)
np.random.seed(CFG.seed)
tf.random.set_seed(CFG.seed)

display(pd.DataFrame([asdict(CFG)]))

## 2. Download dos datasets

In [ ]:
ASVP = Path(
    kagglehub.dataset_download(
        (
            "dejolilandry/"
            "asvpesdspeech-nonspeech-emotional-utterances"
        )
    )
)

GUNSHOT = Path(
    kagglehub.dataset_download(
        "emrahaydemr/gunshot-audio-dataset"
    )
)

URBAN = Path(
    kagglehub.dataset_download(
        "chrisfilo/urbansound8k"
    )
)

SCREAM = Path(
    kagglehub.dataset_download(
        (
            "aananehsansiam/"
            "audio-dataset-of-scream-and-non-scream"
        )
    )
)

print("ASVP:", ASVP)
print("Gunshot:", GUNSHOT)
print("UrbanSound8K:", URBAN)
print("Scream:", SCREAM)

In [ ]:
import time


def zip_valido(
    caminho: Path,
) -> bool:
    return (
        caminho.exists()
        and caminho.stat().st_size > 0
        and zipfile.is_zipfile(caminho)
    )


def baixar_zip(
    urls: str | list[str],
    destino: Path,
    obrigatorio: bool = True,
    tentativas_por_url: int = 4,
) -> bool:
    if zip_valido(destino):
        print("Já disponível:", destino)
        return True

    if isinstance(urls, str):
        urls = [urls]

    destino.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporario = destino.with_suffix(
        destino.suffix + ".part"
    )

    # Pode ter terminado o download, mas não ter sido renomeado.
    if zip_valido(temporario):
        temporario.replace(destino)
        print("Download parcial já estava completo:", destino)
        return True

    ultimo_erro = None

    for url in urls:
        for tentativa in range(
            1,
            tentativas_por_url + 1,
        ):
            try:
                tamanho_existente = (
                    temporario.stat().st_size
                    if temporario.exists()
                    else 0
                )

                cabecalhos = {
                    "User-Agent": (
                        "Mozilla/5.0 "
                        "violence-edge-ai-aed-v8"
                    )
                }

                if tamanho_existente > 0:
                    cabecalhos["Range"] = (
                        f"bytes={tamanho_existente}-"
                    )

                print(
                    "\nBaixando:",
                    destino.name,
                )
                print(
                    "Rota:",
                    url,
                )
                print(
                    f"Tentativa: {tentativa}/"
                    f"{tentativas_por_url}",
                )
                print(
                    "Já baixado:",
                    f"{tamanho_existente / 1024**2:.1f} MB",
                )

                with requests.get(
                    url,
                    stream=True,
                    timeout=(45, 600),
                    verify=True,
                    allow_redirects=True,
                    headers=cabecalhos,
                ) as resposta:
                    # O servidor pode informar que o arquivo já terminou.
                    if (
                        resposta.status_code == 416
                        and zip_valido(temporario)
                    ):
                        temporario.replace(destino)
                        return True

                    resposta.raise_for_status()

                    # 206 significa que o servidor aceitou continuar.
                    continuar = (
                        tamanho_existente > 0
                        and resposta.status_code == 206
                    )

                    if continuar:
                        modo = "ab"
                        inicio_barra = tamanho_existente
                    else:
                        modo = "wb"
                        inicio_barra = 0

                    restante = int(
                        resposta.headers.get(
                            "content-length",
                            0,
                        )
                    )

                    total = (
                        inicio_barra + restante
                        if restante > 0
                        else None
                    )

                    with temporario.open(
                        modo
                    ) as arquivo:
                        with tqdm(
                            total=total,
                            initial=inicio_barra,
                            unit="B",
                            unit_scale=True,
                            desc=destino.name,
                        ) as barra:
                            for bloco in resposta.iter_content(
                                chunk_size=4 * 1024 * 1024
                            ):
                                if not bloco:
                                    continue

                                arquivo.write(bloco)
                                barra.update(len(bloco))

                if not zip_valido(temporario):
                    raise RuntimeError(
                        "O arquivo recebido não é um ZIP válido."
                    )

                temporario.replace(destino)

                print(
                    "Download concluído:",
                    destino,
                )

                return True

            except Exception as erro:
                ultimo_erro = erro

                print(
                    "Falhou:",
                    type(erro).__name__,
                    erro,
                )

                # Mantém o .part para tentar continuar depois.
                espera = min(
                    60,
                    5 * (2 ** (tentativa - 1)),
                )

                print(
                    f"Nova tentativa em {espera}s..."
                )

                time.sleep(espera)

    mensagem = (
        f"Não foi possível baixar {destino.name}. "
        f"Último erro: {ultimo_erro}"
    )

    if obrigatorio:
        raise RuntimeError(mensagem)

    print("\nAVISO:", mensagem)
    print(
        "Este dataset será ignorado e "
        "o notebook continuará."
    )

    return False


def extrair_zip(
    arquivo_zip: Path,
    destino: Path,
) -> bool:
    if not zip_valido(arquivo_zip):
        print(
            "Extração ignorada; ZIP ausente:",
            arquivo_zip,
        )
        return False

    marcador = destino / ".extraido"

    if marcador.exists():
        print(
            "Já extraído:",
            destino,
        )
        return True

    destino.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        "Extraindo:",
        arquivo_zip.name,
    )

    with zipfile.ZipFile(
        arquivo_zip
    ) as arquivo:
        arquivo.extractall(destino)

    marcador.touch()

    print(
        "Extração concluída:",
        destino,
    )

    return True


# ============================================================
# ESC-50 — obrigatório
# ============================================================

ZIP_ESC = DADOS / "esc50.zip"

esc_disponivel = baixar_zip(
    [
        (
            "https://github.com/karolpiczak/"
            "ESC-50/archive/refs/heads/master.zip"
        ),
    ],
    ZIP_ESC,
    obrigatorio=True,
)

if esc_disponivel:
    extrair_zip(
        ZIP_ESC,
        ESC_DIR,
    )


# ============================================================
# TUT Rare — opcional
# O notebook continua caso o Zenodo permaneça indisponível.
# ============================================================

TUT_DISPONIVEL = False

if CFG.usar_tut:
    ZIP_TUT = DADOS / "tut_rare.zip"

    nome_tut = (
        "TUT-rare-sound-events-2017-development."
        "source_data_events.zip"
    )

    TUT_DISPONIVEL = baixar_zip(
        [
            (
                "https://zenodo.org/api/records/401395/"
                f"files/{nome_tut}/content"
            ),
            (
                "https://zenodo.org/records/401395/files/"
                f"{nome_tut}?download=1"
            ),
        ],
        ZIP_TUT,
        obrigatorio=False,
        tentativas_por_url=4,
    )

    if TUT_DISPONIVEL:
        extrair_zip(
            ZIP_TUT,
            TUT_DIR,
        )

print(
    "TUT disponível:",
    TUT_DISPONIVEL,
)


# ============================================================
# Dataset-AOB — obrigatório
# ============================================================

if CFG.usar_aob:
    ZIP_AOB_DEV = (
        DADOS / "aob_dev.zip"
    )

    ZIP_AOB_EVAL = (
        DADOS / "aob_eval.zip"
    )

    aob_dev_disponivel = baixar_zip(
        [
            (
                "https://zenodo.org/api/records/4319802/"
                "files/DATASET_AOB.zip/content"
            ),
            (
                "https://zenodo.org/records/4319802/files/"
                "DATASET_AOB.zip?download=1"
            ),
        ],
        ZIP_AOB_DEV,
        obrigatorio=True,
    )

    aob_eval_disponivel = baixar_zip(
        [
            (
                "https://zenodo.org/api/records/4319802/"
                "files/EVALUATION_dataset_AOB.zip/content"
            ),
            (
                "https://zenodo.org/records/4319802/files/"
                "EVALUATION_dataset_AOB.zip?download=1"
            ),
        ],
        ZIP_AOB_EVAL,
        obrigatorio=True,
    )

    if aob_dev_disponivel:
        extrair_zip(
            ZIP_AOB_DEV,
            AOB_DEV,
        )

    if aob_eval_disponivel:
        extrair_zip(
            ZIP_AOB_EVAL,
            AOB_EVAL,
        )


# ============================================================
# TAU Privacy — obrigatório
# ============================================================

if CFG.usar_tau_privacy:
    ZIP_TAU = (
        DADOS / "tau_privacy.zip"
    )

    tau_disponivel = baixar_zip(
        [
            (
                "https://zenodo.org/api/records/10378145/"
                "files/final_data_wav_split.zip/content"
            ),
            (
                "https://zenodo.org/records/10378145/files/"
                "final_data_wav_split.zip?download=1"
            ),
        ],
        ZIP_TAU,
        obrigatorio=True,
    )

    if tau_disponivel:
        extrair_zip(
            ZIP_TAU,
            TAU_DIR,
        )


print("\nDownloads concluídos.")
print("ESC-50:", esc_disponivel)
print("TUT Rare:", TUT_DISPONIVEL)

if CFG.usar_aob:
    print("AOB desenvolvimento:", aob_dev_disponivel)
    print("AOB avaliação:", aob_eval_disponivel)

if CFG.usar_tau_privacy:
    print("TAU Privacy:", tau_disponivel)

## 3. Reforço manual opcional

## 4. Metadados unificados

In [ ]:
EXTENSOES = {
    ".wav",
    ".mp3",
    ".ogg",
    ".flac",
    ".m4a",
}


def audios_em(
    pasta: Path | None,
) -> list[Path]:
    if pasta is None or not pasta.exists():
        return []

    return sorted(
        caminho
        for caminho in pasta.rglob("*")
        if (
            caminho.is_file()
            and caminho.suffix.lower()
            in EXTENSOES
        )
    )


def token(texto: str) -> str:
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(texto).lower(),
    ).strip("_")


def base_gravacao(nome: str) -> str:
    nome = token(nome)
    nome = re.sub(
        r"_(speech|no_speech|nospeech).*$",
        "",
        nome,
    )
    nome = re.sub(
        r"_(chunk|segment|slice|part)_?\d+$",
        "",
        nome,
    )
    return nome or "gravacao"


def adicionar(
    registros: list[dict[str, Any]],
    arquivo: Path,
    classe: str,
    fonte: str,
    grupo: str,
    split_fixo: str = "",
    extra: str = "",
) -> None:
    arquivo = Path(arquivo)

    if arquivo.exists() and classe in ID:
        registros.append(
            {
                "arquivo": str(arquivo.resolve()),
                "classe": classe,
                "classe_id": ID[classe],
                "familia": FAMILIA[classe],
                "perigo": int(
                    classe != "ambiente_normal"
                ),
                "fonte": fonte,
                "grupo": str(grupo),
                "split_fixo": split_fixo,
                "extra": str(extra),
            }
        )


registros = []


# ASVP
mapa_asvp = {
    "02": "ambiente_normal",
    "04": "choro_distress",
    "06": "grito_panico",
}

for arquivo in audios_em(ASVP):
    numeros = re.findall(
        r"\d+",
        arquivo.stem,
    )

    if len(numeros) < 6:
        continue

    emocao = numeros[2].zfill(2)
    ator = numeros[5].zfill(3)
    classe = mapa_asvp.get(emocao)

    if classe is None:
        continue

    adicionar(
        registros,
        arquivo,
        classe,
        "ASVP",
        f"asvp:ator_{ator}",
        extra=emocao,
    )


# Scream / non-scream
for arquivo in audios_em(SCREAM):
    relativo = arquivo.relative_to(
        SCREAM
    )

    pastas = [
        token(parte)
        for parte in relativo.parts[:-1]
    ]

    if any(
        nome in {
            "non_scream",
            "nonscream",
            "no_scream",
            "not_scream",
        }
        for nome in pastas
    ):
        classe = "ambiente_normal"

    elif any(
        nome in {
            "scream",
            "screams",
        }
        for nome in pastas
    ):
        classe = "grito_panico"

    else:
        continue

    adicionar(
        registros,
        arquivo,
        classe,
        "ScreamDataset",
        (
            f"scream:{classe}:"
            f"{token(arquivo.stem)}"
        ),
        extra="/".join(pastas),
    )


# Gunshot
for arquivo in audios_em(GUNSHOT):
    relativo = arquivo.relative_to(
        GUNSHOT
    )

    origem = token(
        relativo.parts[0]
        if relativo.parts
        else arquivo.parent.name
    )

    sessao = re.sub(
        r"\s*\(\d+\)$",
        "",
        arquivo.stem,
    )
    sessao = token(sessao)

    adicionar(
        registros,
        arquivo,
        "tiro",
        "GunshotKaggle",
        f"gunshot:{origem}:{sessao}",
        extra=origem,
    )


# ESC-50
csvs_esc = list(
    ESC_DIR.rglob("esc50.csv")
)
pastas_esc = [
    pasta
    for pasta in ESC_DIR.rglob("audio")
    if pasta.is_dir()
]

if not csvs_esc or not pastas_esc:
    raise RuntimeError(
        "ESC-50 incompleto."
    )

tabela_esc = pd.read_csv(
    csvs_esc[0]
)
audio_esc = pastas_esc[0]

mapa_esc = {
    "crying_baby": "choro_distress",
    "glass_breaking": "vidro_quebrando",
    "door_wood_knock": "impacto",
}

normais_esc = {
    "clapping",
    "hand_saw",
    "door_wood_creaks",
    "fireworks",
    "chainsaw",
    "siren",
    "engine",
    "vacuum_cleaner",
    "washing_machine",
    "clock_alarm",
    "can_opening",
    "mouse_click",
    "keyboard_typing",
    "footsteps",
    "laughing",
    "breathing",
    "coughing",
    "sneezing",
    "snoring",
    "rain",
    "sea_waves",
    "wind",
    "thunderstorm",
    "airplane",
    "helicopter",
    "train",
}

for linha in tabela_esc.itertuples():
    categoria = str(linha.category)

    if categoria in mapa_esc:
        classe = mapa_esc[categoria]
    elif categoria in normais_esc:
        classe = "ambiente_normal"
    else:
        continue

    adicionar(
        registros,
        audio_esc / str(linha.filename),
        classe,
        "ESC-50",
        f"esc50:{linha.src_file}",
        extra=categoria,
    )


# UrbanSound8K
csvs_urban = list(
    URBAN.rglob("UrbanSound8K.csv")
)

if csvs_urban:
    tabela_urban = pd.read_csv(
        csvs_urban[0]
    )
    raiz_audio = (
        csvs_urban[0].parent.parent
        / "audio"
    )

    for _, linha in tabela_urban.iterrows():
        nome = str(linha["class"])

        if nome == "gun_shot":
            classe = "tiro"
        elif nome in {
            "jackhammer",
            "drilling",
            "siren",
            "car_horn",
            "dog_bark",
            "street_music",
            "children_playing",
            "air_conditioner",
            "engine_idling",
        }:
            classe = "ambiente_normal"
        else:
            continue

        arquivo = (
            raiz_audio
            / f"fold{int(linha['fold'])}"
            / str(linha["slice_file_name"])
        )

        fsid = str(
            linha.get(
                "fsID",
                str(
                    linha["slice_file_name"]
                ).split("-")[0],
            )
        )

        adicionar(
            registros,
            arquivo,
            classe,
            "UrbanSound8K",
            f"urban:fsid_{fsid}",
            extra=nome,
        )


# TUT Rare
if CFG.usar_tut:
    for arquivo in audios_em(TUT_DIR):
        texto = token(str(arquivo))

        if any(
            termo in texto
            for termo in (
                "babycry",
                "baby_cry",
                "babycrying",
            )
        ):
            classe = "choro_distress"
        elif any(
            termo in texto
            for termo in (
                "glassbreak",
                "glass_break",
                "glassbreaking",
            )
        ):
            classe = "vidro_quebrando"
        elif any(
            termo in texto
            for termo in (
                "gunshot",
                "gun_shot",
                "gunfire",
            )
        ):
            classe = "tiro"
        else:
            continue

        adicionar(
            registros,
            arquivo,
            classe,
            "TUT_Rare",
            f"tut:{classe}:{arquivo.stem}",
            extra=classe,
        )


# Dataset-AOB
def classe_aob(
    caminho: Path,
) -> str | None:
    texto = token(str(caminho))

    if any(
        termo in texto
        for termo in (
            "glass_breaking",
            "glassbreaking",
            "glass_break",
        )
    ):
        return "vidro_quebrando"

    if any(
        termo in texto
        for termo in (
            "gun_shot",
            "gunshot",
        )
    ):
        return "tiro"

    if any(
        termo in texto
        for termo in (
            "screams",
            "scream",
        )
    ):
        return "grito_panico"

    if any(
        termo in texto
        for termo in (
            "alarm_siren",
            "children_playing",
            "dog_bark",
            "engine",
            "footsteps",
            "metro_train",
            "rain",
        )
    ):
        return "ambiente_normal"

    return None


if CFG.usar_aob:
    for raiz_aob, split_fixo in (
        (AOB_DEV, ""),
        (AOB_EVAL, "externo"),
    ):
        for arquivo in audios_em(
            raiz_aob
        ):
            classe = classe_aob(
                arquivo
            )

            if classe is None:
                continue

            relativo = arquivo.relative_to(
                raiz_aob
            )

            categoria = token(
                str(relativo.parent)
            )
            grupo_base = base_gravacao(
                relativo.stem
            )

            adicionar(
                registros,
                arquivo,
                classe,
                "Dataset-AOB",
                (
                    f"aob:{split_fixo or 'dev'}:"
                    f"{categoria}:{grupo_base}"
                ),
                split_fixo=split_fixo,
                extra=str(relativo.parent),
            )


# TAU Privacy
def classe_tau(
    caminho: Path,
) -> str | None:
    texto = token(str(caminho))

    if "glass_breaking" in texto:
        return "vidro_quebrando"

    if (
        "gun_shot" in texto
        or "gunshot" in texto
    ):
        return "tiro"

    if "slam" in texto:
        return "impacto"

    if "crying" in texto:
        return "choro_distress"

    if any(
        termo in texto
        for termo in (
            "dog_barking",
            "cough",
            "applause",
            "dished_pot_pan",
            "toilet_flush",
            "cat_meowing",
            "doorbell",
            "drill",
        )
    ):
        return "ambiente_normal"

    return None


if CFG.usar_tau_privacy:
    for arquivo in audios_em(TAU_DIR):
        classe = classe_tau(arquivo)

        if classe is None:
            continue

        relativo = arquivo.relative_to(
            TAU_DIR
        )
        evento = token(
            relativo.parent.parent.name
            if len(relativo.parts) >= 3
            else classe
        )
        base = base_gravacao(
            arquivo.stem
        )

        adicionar(
            registros,
            arquivo,
            classe,
            "TAU-Privacy",
            f"tau:{evento}:{base}",
            extra=str(
                arquivo.relative_to(
                    TAU_DIR
                ).parent
            ),
        )


# Reforço manual
if CFG.usar_reforco_manual:
    for classe in CLASSES:
        for pasta in REFORCO.rglob(
            classe
        ):
            if not pasta.is_dir():
                continue

            for arquivo in audios_em(
                pasta
            ):
                adicionar(
                    registros,
                    arquivo,
                    classe,
                    "ReforcoManual",
                    (
                        f"manual:{classe}:"
                        f"{token(arquivo.stem)}"
                    ),
                    extra=classe,
                )


meta = pd.DataFrame(registros)

if meta.empty:
    raise RuntimeError(
        "Nenhum áudio encontrado."
    )

meta = meta.drop_duplicates(
    subset=["arquivo", "classe"]
)

desenvolvimento = meta[
    meta["split_fixo"] != "externo"
].copy()

externo = meta[
    meta["split_fixo"] == "externo"
].copy()

partes = []

for classe in CLASSES:
    parte = desenvolvimento[
        desenvolvimento["classe"]
        == classe
    ].copy()

    if (
        len(parte)
        > CFG.max_arquivos_por_classe
    ):
        parte = parte.sample(
            CFG.max_arquivos_por_classe,
            random_state=(
                CFG.seed + ID[classe]
            ),
        )

    partes.append(parte)

desenvolvimento = pd.concat(
    partes,
    ignore_index=True,
)

display(
    desenvolvimento.groupby(
        ["classe", "fonte"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        CLASSES,
        fill_value=0,
    )
)

resumo = (
    desenvolvimento.groupby(
        "classe"
    )
    .agg(
        arquivos=("arquivo", "count"),
        grupos=("grupo", "nunique"),
    )
    .reindex(
        CLASSES,
        fill_value=0,
    )
)

display(resumo)

faltantes = [
    classe
    for classe in CLASSES
    if resumo.loc[
        classe,
        "arquivos",
    ] == 0
]

if faltantes:
    raise RuntimeError(
        f"Classes sem dados: {faltantes}"
    )

desenvolvimento.to_csv(
    RESULTADOS
    / "fontes_desenvolvimento.csv",
    index=False,
)

externo.to_csv(
    RESULTADOS
    / "fontes_externo_aob.csv",
    index=False,
)

## 5. Auditoria manual

In [ ]:
for classe in CLASSES:
    print("\n" + "=" * 72)
    print("Classe:", classe)

    parte = desenvolvimento[
        desenvolvimento["classe"]
        == classe
    ]

    for linha in (
        parte.sample(
            min(2, len(parte)),
            random_state=(
                CFG.seed + ID[classe]
            ),
        )
        .itertuples()
    ):
        print(
            linha.fonte,
            "|",
            Path(linha.arquivo).name,
            "|",
            linha.extra,
        )
        display(
            Audio(linha.arquivo)
        )

In [ ]:
APROVAR_AUDITORIA = True

if not APROVAR_AUDITORIA:
    raise RuntimeError(
        "Ouça as amostras. Depois altere "
        "APROVAR_AUDITORIA para True."
    )

## 6. Split global por grupo, sem vazamento

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
def pontuar_split(tabela: pd.DataFrame) -> float:
    alvos = {
        "treino": 0.70,
        "validacao": 0.15,
        "teste": 0.15,
    }

    total_classe = (
        tabela.groupby("classe")
        .size()
        .reindex(CLASSES, fill_value=0)
    )

    score = 0.0

    for split, alvo in alvos.items():
        parte = tabela[
            tabela["split"] == split
        ]

        score += abs(
            len(parte) / len(tabela) - alvo
        )

        contagem = (
            parte.groupby("classe")
            .size()
            .reindex(CLASSES, fill_value=0)
        )

        proporcao = (
            contagem
            / total_classe.replace(0, np.nan)
        ).fillna(0.0)

        score += float(
            np.abs(proporcao - alvo).mean()
        )

    return score


def criar_split_global(
    tabela: pd.DataFrame,
):
    melhor = None
    melhor_score = float("inf")

    for tentativa in range(
        CFG.tentativas_split
    ):
        semente = CFG.seed + tentativa

        divisor_1 = GroupShuffleSplit(
            n_splits=1,
            train_size=0.70,
            random_state=semente,
        )

        idx_treino, idx_tmp = next(
            divisor_1.split(
                tabela,
                groups=tabela["grupo"],
            )
        )

        temporario = tabela.iloc[idx_tmp]

        divisor_2 = GroupShuffleSplit(
            n_splits=1,
            train_size=0.50,
            random_state=semente + 10000,
        )

        idx_val_rel, idx_teste_rel = next(
            divisor_2.split(
                temporario,
                groups=temporario["grupo"],
            )
        )

        candidato = tabela.copy()
        candidato["split"] = ""
        candidato.iloc[
            idx_treino,
            candidato.columns.get_loc("split"),
        ] = "treino"

        indices_val = temporario.index[
            idx_val_rel
        ]
        indices_teste = temporario.index[
            idx_teste_rel
        ]

        candidato.loc[
            indices_val,
            "split",
        ] = "validacao"
        candidato.loc[
            indices_teste,
            "split",
        ] = "teste"

        distribuicao = (
            candidato.groupby(
                ["classe", "split"]
            )
            .size()
            .unstack(fill_value=0)
            .reindex(CLASSES, fill_value=0)
        )

        esperadas = [
            "treino",
            "validacao",
            "teste",
        ]

        if any(
            split not in distribuicao.columns
            for split in esperadas
        ):
            continue

        if (
            distribuicao[esperadas]
            .eq(0)
            .any()
            .any()
        ):
            continue

        score = pontuar_split(candidato)

        if score < melhor_score:
            melhor = candidato
            melhor_score = score

    if melhor is None:
        raise RuntimeError(
            "Não foi possível criar um split válido."
        )

    grupos = {
        split: set(
            melhor.loc[
                melhor["split"] == split,
                "grupo",
            ]
        )
        for split in (
            "treino",
            "validacao",
            "teste",
        )
    }

    intersecoes = {
        "treino_validacao": len(
            grupos["treino"]
            & grupos["validacao"]
        ),
        "treino_teste": len(
            grupos["treino"]
            & grupos["teste"]
        ),
        "validacao_teste": len(
            grupos["validacao"]
            & grupos["teste"]
        ),
    }

    if any(intersecoes.values()):
        raise RuntimeError(
            f"Vazamento entre splits: {intersecoes}"
        )

    return melhor, {
        "score": float(melhor_score),
        "intersecoes": intersecoes,
        "grupos_por_split": {
            split: len(valor)
            for split, valor in grupos.items()
        },
    }


meta_split, auditoria_split = (
    criar_split_global(
        desenvolvimento
    )
)

display(
    meta_split.groupby(
        ["classe", "split"]
    )
    .agg(
        arquivos=("arquivo", "count"),
        grupos=("grupo", "nunique"),
    )
)

print(
    json.dumps(
        auditoria_split,
        indent=2,
    )
)

meta_split.to_csv(
    RESULTADOS / "fontes_com_split.csv",
    index=False,
)

(
    RESULTADOS / "auditoria_split.json"
).write_text(
    json.dumps(
        auditoria_split,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

## 7. Extrair YAMNet + MFCC e ampliar o treino com dados reais

O split de validação e teste permanece o mesmo das fontes-base. FSD50K, Baby Crying Sounds e Donate-a-Cry são adicionados exclusivamente ao treino. IDs do Freesound já usados pelo ESC-50 e UrbanSound8K são removidos do FSD50K para reduzir duplicatas.

In [ ]:
YAMNET_HANDLE = "https://tfhub.dev/google/yamnet/1"
yamnet = hub.load(YAMNET_HANDLE)
AMOSTRAS = int(CFG.sr * CFG.duracao)


def carregar_audio(caminho: str | Path) -> np.ndarray:
    audio, _ = librosa.load(
        str(caminho),
        sr=CFG.sr,
        mono=True,
    )
    audio = np.nan_to_num(audio.astype(np.float32))
    pico = float(np.max(np.abs(audio)))
    if pico > 1.0:
        audio = audio / pico
    return audio


def carregar_audio_bytes(conteudo: bytes) -> np.ndarray:
    audio, sr = sf.read(
        io.BytesIO(conteudo),
        dtype="float32",
        always_2d=False,
    )
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != CFG.sr:
        audio = librosa.resample(
            audio,
            orig_sr=sr,
            target_sr=CFG.sr,
        )
    audio = np.nan_to_num(audio.astype(np.float32))
    pico = float(np.max(np.abs(audio)))
    if pico > 1.0:
        audio = audio / pico
    return audio


def ajustar(audio: np.ndarray) -> np.ndarray:
    if len(audio) < AMOSTRAS:
        return np.pad(audio, (0, AMOSTRAS - len(audio)))
    return audio[:AMOSTRAS]


def janelas_uniformes(
    audio: np.ndarray,
    quantidade: int | None = None,
) -> list[np.ndarray]:
    quantidade = quantidade or CFG.max_janelas_por_arquivo
    if len(audio) <= AMOSTRAS:
        return [ajustar(audio)]
    inicios = np.linspace(
        0,
        len(audio) - AMOSTRAS,
        quantidade,
        dtype=int,
    )
    return [
        audio[inicio:inicio + AMOSTRAS]
        for inicio in inicios
    ]


def janelas_onset(
    audio: np.ndarray,
    quantidade: int | None = None,
) -> list[np.ndarray]:
    quantidade = quantidade or CFG.max_janelas_por_arquivo
    if len(audio) <= AMOSTRAS:
        return [ajustar(audio)]

    onset = librosa.onset.onset_strength(
        y=audio,
        sr=CFG.sr,
        hop_length=CFG.hop_length,
    )
    picos = librosa.util.peak_pick(
        onset,
        pre_max=3,
        post_max=3,
        pre_avg=5,
        post_avg=5,
        delta=0.18,
        wait=8,
    )

    if len(picos) == 0:
        return janelas_uniformes(audio, quantidade)

    ordem = np.argsort(onset[picos])[::-1]
    janelas = []
    usados = []

    for indice in ordem:
        centro = librosa.frames_to_samples(
            int(picos[indice]),
            hop_length=CFG.hop_length,
        )
        inicio = int(
            np.clip(
                centro - AMOSTRAS // 2,
                0,
                len(audio) - AMOSTRAS,
            )
        )
        if any(
            abs(inicio - anterior) < CFG.sr * 0.35
            for anterior in usados
        ):
            continue
        janelas.append(audio[inicio:inicio + AMOSTRAS])
        usados.append(inicio)
        if len(janelas) >= quantidade:
            break

    return janelas or janelas_uniformes(audio, quantidade)


def extrair_vetor(audio: np.ndarray) -> np.ndarray:
    audio = ajustar(audio).astype(np.float32)

    _, embeddings, _ = yamnet(
        tf.convert_to_tensor(audio, dtype=tf.float32)
    )
    emb = embeddings.numpy()

    yamnet_stats = np.concatenate([
        emb.mean(axis=0),
        emb.std(axis=0),
        emb.max(axis=0),
        np.percentile(emb, 90, axis=0),
    ])

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=CFG.sr,
        n_mfcc=CFG.n_mfcc,
        n_fft=CFG.n_fft,
        hop_length=CFG.hop_length,
    )
    delta = librosa.feature.delta(mfcc, width=9)
    delta2 = librosa.feature.delta(mfcc, order=2, width=9)

    rms = librosa.feature.rms(y=audio)
    centroide = librosa.feature.spectral_centroid(
        y=audio,
        sr=CFG.sr,
    )
    largura = librosa.feature.spectral_bandwidth(
        y=audio,
        sr=CFG.sr,
    )
    zcr = librosa.feature.zero_crossing_rate(audio)
    onset = librosa.onset.onset_strength(
        y=audio,
        sr=CFG.sr,
        hop_length=CFG.hop_length,
    )

    mfcc_stats = np.concatenate([
        mfcc.mean(axis=1),
        mfcc.std(axis=1),
        np.percentile(mfcc, [10, 50, 90], axis=1).reshape(-1),
        delta.mean(axis=1),
        delta.std(axis=1),
        delta2.mean(axis=1),
        delta2.std(axis=1),
    ])

    espectral = np.asarray([
        rms.mean(),
        rms.std(),
        rms.max(),
        centroide.mean(),
        centroide.std(),
        largura.mean(),
        largura.std(),
        zcr.mean(),
        zcr.std(),
        onset.mean(),
        onset.std(),
        onset.max(),
        np.max(np.abs(audio)),
    ], dtype=np.float32)

    return np.concatenate([
        yamnet_stats,
        mfcc_stats,
        espectral,
    ]).astype(np.float32)


registros_features = []
vetores = []


def adicionar_janelas(
    audio: np.ndarray,
    classe: str,
    grupo: str,
    fonte: str,
    arquivo: str,
    split: str = "treino",
    quantidade: int | None = None,
):
    if classe in RARAS:
        janelas = janelas_onset(audio, quantidade)
    else:
        janelas = janelas_uniformes(audio, quantidade)

    for janela in janelas:
        vetores.append(extrair_vetor(janela))
        registros_features.append({
            "classe": classe,
            "classe_id": ID[classe],
            "familia": FAMILIA[classe],
            "perigo": int(classe != "ambiente_normal"),
            "grupo": grupo,
            "split": split,
            "fonte": fonte,
            "arquivo": arquivo,
            "suplementar": split == "treino" and fonte in {
                "FSD50K",
                "BabyCryingKaggle",
                "DonateACry",
            },
        })


# Fontes-base, sem augmentation offline.
for linha in tqdm(
    meta_split.itertuples(),
    total=len(meta_split),
    desc="Fontes-base",
):
    try:
        audio = carregar_audio(linha.arquivo)
        adicionar_janelas(
            audio=audio,
            classe=linha.classe,
            grupo=linha.grupo,
            fonte=linha.fonte,
            arquivo=linha.arquivo,
            split=linha.split,
            quantidade=CFG.max_janelas_por_arquivo,
        )
    except Exception as erro:
        print("Falha base:", linha.arquivo, erro)

print("Janelas-base:", len(registros_features))

## 8. FSD50K por streaming

In [ ]:
def normalizar_label_fsd(texto: str) -> str:
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(texto).lower(),
    ).strip("_")


def mapear_labels_fsd(labels: list[str]) -> str | None:
    nomes = [normalizar_label_fsd(label) for label in labels]
    candidatos = set()

    if any("gunshot" in nome or "gunfire" in nome for nome in nomes):
        candidatos.add("tiro")

    if any("glass" in nome or "shatter" in nome for nome in nomes):
        candidatos.add("vidro_quebrando")

    if any("screaming" in nome or nome == "scream" for nome in nomes):
        candidatos.add("grito_panico")

    if any(
        "crying" in nome
        or "sobbing" in nome
        or "baby_cry" in nome
        or "infant_cry" in nome
        for nome in nomes
    ):
        candidatos.add("choro_distress")

    termos_impacto = (
        "knock",
        "slam",
        "thump",
        "thud",
        "punch",
        "smash",
        "crash",
        "crack",
        "impact",
    )

    if any(
        any(termo in nome for termo in termos_impacto)
        for nome in nomes
    ):
        candidatos.add("impacto")

    # Multi-rótulo ambíguo entre classes-alvo é descartado.
    if len(candidatos) == 1:
        return next(iter(candidatos))

    if len(candidatos) > 1:
        return None

    negativos = (
        "clapping",
        "applause",
        "fireworks",
        "hammer",
        "drill",
        "chainsaw",
        "siren",
        "engine",
        "speech",
        "laughter",
        "cough",
        "bark",
        "music",
        "door",
        "wood",
        "metal",
        "footsteps",
        "vacuum",
    )

    if any(
        any(termo in nome for termo in negativos)
        for nome in nomes
    ):
        return "ambiente_normal"

    return None


if CFG.usar_fsd50k:
    caminho_dev = Path(
        hf_hub_download(
            repo_id="Fhrozen/FSD50k",
            repo_type="dataset",
            filename="labels/dev.csv",
        )
    )

    tabela_fsd = pd.read_csv(caminho_dev)

    # Usa somente o split oficial train do FSD50K.
    if "split" in tabela_fsd.columns:
        tabela_fsd = tabela_fsd[
            tabela_fsd["split"].astype(str).str.lower() == "train"
        ].copy()

    ids_reservados = set()

    for grupo in meta_split["grupo"].astype(str):
        if grupo.startswith("esc50:src_"):
            ids_reservados.add(grupo.split("esc50:src_", 1)[1])
        elif grupo.startswith("urban:fsid_"):
            ids_reservados.add(grupo.split("urban:fsid_", 1)[1])

    selecoes = {classe: [] for classe in CLASSES}

    for linha in tabela_fsd.itertuples():
        identificador = str(linha.fname)

        if identificador in ids_reservados:
            continue

        classe = mapear_labels_fsd(
            str(linha.labels).split(",")
        )

        if classe is not None:
            selecoes[classe].append(identificador)

    rng_fsd = random.Random(CFG.seed + 500)

    for classe, ids in selecoes.items():
        ids = list(dict.fromkeys(ids))
        rng_fsd.shuffle(ids)
        limite = (
            CFG.max_fsd_normal
            if classe == "ambiente_normal"
            else CFG.max_fsd_por_classe
        )
        selecoes[classe] = ids[:limite]

    mapa_fsd = {
        identificador: classe
        for classe, ids in selecoes.items()
        for identificador in ids
    }

    print(
        "FSD50K selecionado:",
        {classe: len(ids) for classe, ids in selecoes.items()},
    )

    pendentes = set(mapa_fsd)

    fluxo = load_dataset(
        "philgzl/fsd50k",
        split="dev",
        streaming=True,
        features=Features({
            "audio": Value("binary"),
            "name": Value("string"),
        }),
    )

    barra = tqdm(total=len(pendentes), desc="FSD50K")

    for item in fluxo:
        identificador = Path(str(item["name"])).stem.split("_")[0]

        if identificador not in pendentes:
            continue

        classe = mapa_fsd[identificador]

        try:
            audio = carregar_audio_bytes(item["audio"])
            adicionar_janelas(
                audio=audio,
                classe=classe,
                grupo=f"fsd50k:{identificador}",
                fonte="FSD50K",
                arquivo=f"stream://{identificador}",
                split="treino",
                quantidade=CFG.max_janelas_suplementares,
            )
        except Exception as erro:
            print("Falha FSD50K:", identificador, erro)

        pendentes.remove(identificador)
        barra.update(1)

        if not pendentes:
            break

    barra.close()
    print("FSD50K não encontrados:", len(pendentes))
else:
    print("FSD50K desativado.")

## 9. Reforços reais de choro

In [ ]:
BABY_KAGGLE = None

if CFG.usar_baby_kaggle:
    try:
        BABY_KAGGLE = Path(
            kagglehub.dataset_download(
                "mennaahmed23/baby-crying-sounds-dataset"
            )
        )
    except Exception as erro:
        print("Baby Crying Kaggle indisponível:", erro)


if BABY_KAGGLE is not None:
    arquivos = []

    for caminho in BABY_KAGGLE.rglob("*"):
        if not caminho.is_file() or caminho.suffix.lower() not in EXTENSOES:
            continue

        texto = normalizar_label_fsd(str(caminho))

        if any(
            termo in texto
            for termo in (
                "non_cry",
                "noncry",
                "no_cry",
                "not_cry",
                "silence",
            )
        ):
            continue

        arquivos.append(caminho)

    random.Random(CFG.seed + 600).shuffle(arquivos)
    arquivos = arquivos[:CFG.max_baby_kaggle]

    for caminho in tqdm(arquivos, desc="Baby Crying Kaggle"):
        try:
            audio = carregar_audio(caminho)
            adicionar_janelas(
                audio=audio,
                classe="choro_distress",
                grupo=(
                    "baby_kaggle:"
                    + normalizar_label_fsd(caminho.stem)
                ),
                fonte="BabyCryingKaggle",
                arquivo=str(caminho),
                split="treino",
                quantidade=CFG.max_janelas_suplementares,
            )
        except Exception as erro:
            print("Falha baby Kaggle:", caminho, erro)


if CFG.usar_donate_cry:
    try:
        if not DONATE_DIR.exists():
            subprocess.run(
                [
                    "git",
                    "clone",
                    "--depth",
                    "1",
                    "https://github.com/gveres/donateacry-corpus.git",
                    str(DONATE_DIR),
                ],
                check=True,
            )

        arquivos = [
            caminho
            for caminho in DONATE_DIR.rglob("*")
            if caminho.is_file()
            and caminho.suffix.lower() in EXTENSOES
        ]

        random.Random(CFG.seed + 700).shuffle(arquivos)
        arquivos = arquivos[:CFG.max_donate_cry]

        for caminho in tqdm(arquivos, desc="Donate-a-Cry"):
            try:
                audio = carregar_audio(caminho)
                bebe = normalizar_label_fsd(
                    caminho.stem.split("-")[0]
                )
                adicionar_janelas(
                    audio=audio,
                    classe="choro_distress",
                    grupo=f"donate_cry:{bebe}",
                    fonte="DonateACry",
                    arquivo=str(caminho),
                    split="treino",
                    quantidade=CFG.max_janelas_suplementares,
                )
            except Exception as erro:
                print("Falha Donate-a-Cry:", caminho, erro)

    except Exception as erro:
        print("Donate-a-Cry indisponível:", erro)


X = np.asarray(vetores, dtype=np.float32)
tabela_features = pd.DataFrame(registros_features)

np.save(CACHE / "features.npy", X)
tabela_features.to_csv(
    CACHE / "features.csv",
    index=False,
)

print("Features:", X.shape)
print(f"Memória: {X.nbytes / 1024**3:.2f} GB")

display(
    tabela_features.groupby(
        ["classe", "split", "fonte"]
    )
    .size()
    .to_frame("janelas")
)

display(
    tabela_features.groupby(
        ["classe", "split"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(CLASSES, fill_value=0)
)

## 10. Normalização e datasets balanceados

In [ ]:
indices_treino = np.where(
    tabela_features["split"].to_numpy() == "treino"
)[0]
indices_validacao = np.where(
    tabela_features["split"].to_numpy() == "validacao"
)[0]
indices_teste = np.where(
    tabela_features["split"].to_numpy() == "teste"
)[0]

scaler = StandardScaler()

X_treino = scaler.fit_transform(
    X[indices_treino]
).astype(np.float32)
X_validacao = scaler.transform(
    X[indices_validacao]
).astype(np.float32)
X_teste = scaler.transform(
    X[indices_teste]
).astype(np.float32)

np.savez_compressed(
    MODELOS / "normalizacao.npz",
    media=scaler.mean_.astype(np.float32),
    escala=scaler.scale_.astype(np.float32),
)

tabela_treino = (
    tabela_features.iloc[indices_treino]
    .copy()
    .reset_index(drop=True)
)
tabela_validacao = (
    tabela_features.iloc[indices_validacao]
    .copy()
    .reset_index(drop=True)
)
tabela_teste = (
    tabela_features.iloc[indices_teste]
    .copy()
    .reset_index(drop=True)
)


def construir_rotulos(tabela: pd.DataFrame):
    evento = tabela["classe_id"].astype(np.int32).to_numpy()
    familia = tabela["familia"].astype(np.int32).to_numpy()
    perigo = tabela["perigo"].astype(np.float32).to_numpy()[:, None]

    vocal = np.zeros(len(tabela), dtype=np.int32)
    transitorio = np.zeros(len(tabela), dtype=np.int32)
    peso_vocal = np.zeros(len(tabela), dtype=np.float32)
    peso_transitorio = np.zeros(len(tabela), dtype=np.float32)

    for indice, classe in enumerate(tabela["classe"].tolist()):
        if classe in VOCAL:
            vocal[indice] = VOCAL[classe]
            peso_vocal[indice] = 1.0
        if classe in TRANSITORIO:
            transitorio[indice] = TRANSITORIO[classe]
            peso_transitorio[indice] = 1.0

    y = {
        "evento": evento,
        "familia": familia,
        "vocal": vocal,
        "transitorio": transitorio,
        "perigo": perigo,
    }

    pesos = {
        "evento": np.ones(len(tabela), dtype=np.float32),
        "familia": np.ones(len(tabela), dtype=np.float32),
        "vocal": peso_vocal,
        "transitorio": peso_transitorio,
        "perigo": np.ones(len(tabela), dtype=np.float32),
    }

    return y, pesos


y_treino, pesos_treino = construir_rotulos(tabela_treino)
y_validacao, pesos_validacao = construir_rotulos(tabela_validacao)
y_teste, pesos_teste = construir_rotulos(tabela_teste)


def dataset_finito(X_dados, y, pesos):
    return (
        tf.data.Dataset
        .from_tensor_slices((X_dados, y, pesos))
        .batch(CFG.lote)
        .prefetch(1)
    )


def dataset_balanceado(
    X_dados,
    tabela,
    semente,
    indices_extras=None,
):
    datasets = []
    pesos_classes = []

    for classe in CLASSES:
        indices = np.where(
            tabela["classe_id"].to_numpy() == ID[classe]
        )[0]

        if (
            indices_extras is not None
            and classe in indices_extras
            and len(indices_extras[classe]) > 0
        ):
            indices = np.concatenate([
                indices,
                indices_extras[classe],
            ])

        parte = tabela.iloc[indices]
        y_parte, pesos_parte = construir_rotulos(parte)

        dataset_classe = (
            tf.data.Dataset
            .from_tensor_slices(
                (X_dados[indices], y_parte, pesos_parte)
            )
            .shuffle(
                min(len(indices), 6000),
                seed=semente + ID[classe],
                reshuffle_each_iteration=True,
            )
            .repeat()
        )

        datasets.append(dataset_classe)
        pesos_classes.append(PESOS_AMOSTRAGEM[classe])

    return (
        tf.data.Dataset
        .sample_from_datasets(
            datasets,
            weights=pesos_classes,
            seed=semente,
            stop_on_empty_dataset=False,
        )
        .batch(CFG.lote)
        .prefetch(1)
    )


ds_treino_1 = dataset_balanceado(
    X_treino,
    tabela_treino,
    CFG.seed,
)
ds_validacao = dataset_finito(
    X_validacao,
    y_validacao,
    pesos_validacao,
)

passos_epoca = int(
    np.clip(
        math.ceil(len(X_treino) / CFG.lote),
        450,
        1050,
    )
)

print("Treino:", X_treino.shape)
print("Validação:", X_validacao.shape)
print("Teste:", X_teste.shape)
print("Passos por época:", passos_epoca)

del X
gc.collect()

## 11. Modelo hierárquico com focal loss

In [ ]:
@tf.keras.utils.register_keras_serializable(
    package="ViolenceEdgeAI"
)
class SparseFocalLoss(tf.keras.losses.Loss):
    def __init__(
        self,
        gamma=1.5,
        alpha=None,
        name="sparse_focal_loss",
    ):
        super().__init__(name=name)
        self.gamma = float(gamma)
        self.alpha = (
            None
            if alpha is None
            else [float(valor) for valor in alpha]
        )

    def call(self, y_true, y_pred):
        y_true = tf.cast(
            tf.reshape(y_true, [-1]),
            tf.int32,
        )
        y_pred = tf.clip_by_value(
            y_pred,
            1e-7,
            1.0 - 1e-7,
        )
        indices = tf.stack([
            tf.range(tf.shape(y_true)[0]),
            y_true,
        ], axis=1)
        p_t = tf.gather_nd(y_pred, indices)
        perda = -tf.pow(1.0 - p_t, self.gamma) * tf.math.log(p_t)

        if self.alpha is not None:
            alpha = tf.convert_to_tensor(
                self.alpha,
                dtype=tf.float32,
            )
            perda *= tf.gather(alpha, y_true)

        return perda

    def get_config(self):
        return {
            "gamma": self.gamma,
            "alpha": self.alpha,
            "name": self.name,
        }


def criar_modelo():
    entrada = tf.keras.Input(
        shape=(X_treino.shape[1],),
        dtype=tf.float32,
        name="features",
    )

    x = tf.keras.layers.GaussianNoise(0.012)(entrada)
    x = tf.keras.layers.Dense(
        1024,
        activation="swish",
        kernel_regularizer=tf.keras.regularizers.l2(2e-5),
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.40)(x)

    residual = tf.keras.layers.Dense(512)(x)
    x = tf.keras.layers.Dense(
        512,
        activation="swish",
        kernel_regularizer=tf.keras.regularizers.l2(2e-5),
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.32)(x)
    x = tf.keras.layers.Dense(512)(x)
    x = tf.keras.layers.Add()([x, residual])
    x = tf.keras.layers.Activation("swish")(x)

    x = tf.keras.layers.Dense(
        256,
        activation="swish",
        kernel_regularizer=tf.keras.regularizers.l2(1e-5),
    )(x)
    x = tf.keras.layers.Dropout(0.25)(x)

    saidas = {
        "evento": tf.keras.layers.Dense(
            len(CLASSES),
            activation="softmax",
            name="evento",
        )(x),
        "familia": tf.keras.layers.Dense(
            3,
            activation="softmax",
            name="familia",
        )(x),
        "vocal": tf.keras.layers.Dense(
            2,
            activation="softmax",
            name="vocal",
        )(x),
        "transitorio": tf.keras.layers.Dense(
            3,
            activation="softmax",
            name="transitorio",
        )(x),
        "perigo": tf.keras.layers.Dense(
            1,
            activation="sigmoid",
            name="perigo",
        )(x),
    }

    return tf.keras.Model(
        entrada,
        saidas,
        name="aed_v8_hierarquico",
    )


def compilar_modelo(modelo, taxa):
    modelo.compile(
        optimizer=tf.keras.optimizers.AdamW(
            learning_rate=taxa,
            weight_decay=1e-5,
            clipnorm=1.0,
        ),
        loss={
            "evento": SparseFocalLoss(
                gamma=1.4,
                alpha=[0.85, 1.0, 1.18, 1.38, 1.08, 1.0],
            ),
            "familia": SparseFocalLoss(gamma=1.2),
            "vocal": SparseFocalLoss(
                gamma=1.3,
                alpha=[1.0, 1.18],
            ),
            "transitorio": SparseFocalLoss(
                gamma=1.3,
                alpha=[1.30, 1.05, 1.0],
            ),
            "perigo": "binary_crossentropy",
        },
        loss_weights={
            "evento": 1.0,
            "familia": 0.25,
            "vocal": 0.32,
            "transitorio": 0.38,
            "perigo": 0.32,
        },
        metrics={
            "evento": [
                tf.keras.metrics.SparseCategoricalAccuracy(
                    name="accuracy"
                )
            ],
            "familia": [
                tf.keras.metrics.SparseCategoricalAccuracy(
                    name="accuracy"
                )
            ],
            "vocal": [
                tf.keras.metrics.SparseCategoricalAccuracy(
                    name="accuracy"
                )
            ],
            "transitorio": [
                tf.keras.metrics.SparseCategoricalAccuracy(
                    name="accuracy"
                )
            ],
            "perigo": [
                tf.keras.metrics.BinaryAccuracy(name="accuracy"),
                tf.keras.metrics.AUC(name="auc"),
            ],
        },
        jit_compile=False,
    )


modelo = criar_modelo()
compilar_modelo(modelo, 3e-4)
modelo.summary()

## 12. Macro-F1 hierárquico

In [ ]:
PESOS_INICIAIS = {
    "evento": 0.48,
    "familia": 0.17,
    "ramo": 0.27,
    "perigo": 0.08,
}


def combinar_saidas(saidas, pesos):
    evento = np.asarray(saidas["evento"], dtype=np.float64)
    familia = np.asarray(saidas["familia"], dtype=np.float64)
    vocal = np.asarray(saidas["vocal"], dtype=np.float64)
    transitorio = np.asarray(saidas["transitorio"], dtype=np.float64)
    perigo = np.asarray(saidas["perigo"], dtype=np.float64).reshape(-1)

    eps = 1e-8
    scores = np.zeros_like(evento)

    scores[:, ID["ambiente_normal"]] = (
        np.power(evento[:, 0] + eps, pesos["evento"])
        * np.power(familia[:, 0] + eps, pesos["familia"])
        * np.power(1.0 - perigo + eps, pesos["perigo"])
    )

    scores[:, ID["grito_panico"]] = (
        np.power(evento[:, 1] + eps, pesos["evento"])
        * np.power(familia[:, 1] + eps, pesos["familia"])
        * np.power(vocal[:, 0] + eps, pesos["ramo"])
        * np.power(perigo + eps, pesos["perigo"])
    )

    scores[:, ID["choro_distress"]] = (
        np.power(evento[:, 2] + eps, pesos["evento"])
        * np.power(familia[:, 1] + eps, pesos["familia"])
        * np.power(vocal[:, 1] + eps, pesos["ramo"])
        * np.power(perigo + eps, pesos["perigo"])
    )

    for classe, indice_ramo in TRANSITORIO.items():
        indice = ID[classe]
        scores[:, indice] = (
            np.power(evento[:, indice] + eps, pesos["evento"])
            * np.power(familia[:, 2] + eps, pesos["familia"])
            * np.power(
                transitorio[:, indice_ramo] + eps,
                pesos["ramo"],
            )
            * np.power(perigo + eps, pesos["perigo"])
        )

    scores /= scores.sum(axis=1, keepdims=True) + eps
    return scores


class SalvarMelhorMacroF1(tf.keras.callbacks.Callback):
    def __init__(self, caminho, X_val, y_val, pesos):
        super().__init__()
        self.caminho = Path(caminho)
        self.X_val = X_val
        self.y_val = y_val
        self.pesos = pesos
        self.melhor = -np.inf

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        saidas = self.model.predict(
            self.X_val,
            batch_size=256,
            verbose=0,
        )
        scores = combinar_saidas(saidas, self.pesos)
        previsto = np.argmax(scores, axis=1)
        macro_f1 = f1_score(
            self.y_val,
            previsto,
            average="macro",
            zero_division=0,
        )
        logs["val_macro_f1_combinado"] = macro_f1
        print(f" — val_macro_f1_combinado: {macro_f1:.4f}")

        if macro_f1 > self.melhor:
            self.melhor = macro_f1
            self.model.save(self.caminho)
            print("Melhor modelo salvo:", self.caminho)

## 13. Treinamento em duas etapas

In [ ]:
MODELO_1 = MODELOS / "aed_v8_etapa_1.keras"
MODELO_2 = MODELOS / "aed_v8_etapa_2.keras"
MODELO_FINAL = MODELOS / "aed_v8_final.keras"


def callbacks_etapa(caminho, historico, paciencia):
    return [
        SalvarMelhorMacroF1(
            caminho,
            X_validacao,
            y_validacao["evento"],
            PESOS_INICIAIS,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_macro_f1_combinado",
            mode="max",
            patience=paciencia,
            restore_best_weights=False,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.5,
            patience=4,
            min_lr=1e-6,
            verbose=1,
        ),
        tf.keras.callbacks.CSVLogger(
            RESULTADOS / historico
        ),
    ]


modelo.fit(
    ds_treino_1,
    validation_data=ds_validacao,
    epochs=CFG.epocas_etapa_1,
    steps_per_epoch=passos_epoca,
    callbacks=callbacks_etapa(
        MODELO_1,
        "historico_etapa_1.csv",
        CFG.paciencia,
    ),
    verbose=1,
)

modelo_1 = tf.keras.models.load_model(
    MODELO_1,
    custom_objects={"SparseFocalLoss": SparseFocalLoss},
)

# Mineração de erros reais no treino.
saidas_treino = modelo_1.predict(
    X_treino,
    batch_size=256,
    verbose=1,
)
scores_treino = combinar_saidas(
    saidas_treino,
    PESOS_INICIAIS,
)
real_treino = tabela_treino["classe_id"].to_numpy()
previsto_treino = np.argmax(scores_treino, axis=1)
confianca_errada = scores_treino[
    np.arange(len(scores_treino)),
    previsto_treino,
]


def selecionar_dificeis(classe, limite):
    indice = ID[classe]
    candidatos = np.where(
        (real_treino == indice)
        & (previsto_treino != real_treino)
    )[0]

    if len(candidatos) == 0:
        return np.asarray([], dtype=np.int64)

    ordem = np.argsort(
        confianca_errada[candidatos]
    )[::-1]
    selecionados = candidatos[ordem[:limite]]

    return np.repeat(
        selecionados,
        CFG.repeticoes_dificeis,
    )


indices_extras = {
    "ambiente_normal": selecionar_dificeis(
        "ambiente_normal",
        CFG.max_hard_normal,
    ),
    "choro_distress": selecionar_dificeis(
        "choro_distress",
        CFG.max_hard_choro,
    ),
    "impacto": selecionar_dificeis(
        "impacto",
        CFG.max_hard_impacto,
    ),
}

for classe, indices in indices_extras.items():
    print(
        classe,
        "entradas difíceis extras:",
        len(indices),
    )

hard_unicos = np.unique(
    np.concatenate([
        valor
        for valor in indices_extras.values()
        if len(valor) > 0
    ])
) if any(len(v) > 0 for v in indices_extras.values()) else np.asarray([], dtype=int)

if len(hard_unicos) > 0:
    hard_df = tabela_treino.iloc[hard_unicos].copy()
    hard_df["previsto"] = [
        CLASSES[indice]
        for indice in previsto_treino[hard_unicos]
    ]
    hard_df["confianca_errada"] = (
        confianca_errada[hard_unicos]
    )
    hard_df.to_csv(
        RESULTADOS / "exemplos_dificeis.csv",
        index=False,
    )


ds_treino_2 = dataset_balanceado(
    X_treino,
    tabela_treino,
    CFG.seed + 100,
    indices_extras=indices_extras,
)

compilar_modelo(modelo_1, 6e-5)

modelo_1.fit(
    ds_treino_2,
    validation_data=ds_validacao,
    epochs=CFG.epocas_etapa_2,
    steps_per_epoch=passos_epoca,
    callbacks=callbacks_etapa(
        MODELO_2,
        "historico_etapa_2.csv",
        8,
    ),
    verbose=1,
)

modelo_2 = tf.keras.models.load_model(
    MODELO_2,
    custom_objects={"SparseFocalLoss": SparseFocalLoss},
)

## 14. Selecionar etapa e combinação pela validação

In [ ]:
CANDIDATOS_PESOS = [
    {"evento": 0.58, "familia": 0.14, "ramo": 0.22, "perigo": 0.06},
    {"evento": 0.52, "familia": 0.16, "ramo": 0.25, "perigo": 0.07},
    {"evento": 0.48, "familia": 0.17, "ramo": 0.27, "perigo": 0.08},
    {"evento": 0.43, "familia": 0.19, "ramo": 0.29, "perigo": 0.09},
    {"evento": 0.40, "familia": 0.20, "ramo": 0.30, "perigo": 0.10},
]

resultados = []
modelos = {
    "etapa_1": modelo_1,
    "etapa_2": modelo_2,
}

for nome, candidato in modelos.items():
    saidas = candidato.predict(
        X_validacao,
        batch_size=256,
        verbose=0,
    )

    for pesos in CANDIDATOS_PESOS:
        scores = combinar_saidas(saidas, pesos)
        previsto = np.argmax(scores, axis=1)
        resultados.append({
            "etapa": nome,
            "pesos": pesos,
            "accuracy": float(
                accuracy_score(
                    y_validacao["evento"],
                    previsto,
                )
            ),
            "balanced_accuracy": float(
                balanced_accuracy_score(
                    y_validacao["evento"],
                    previsto,
                )
            ),
            "f1_macro": float(
                f1_score(
                    y_validacao["evento"],
                    previsto,
                    average="macro",
                    zero_division=0,
                )
            ),
        })

comparacao = (
    pd.DataFrame(resultados)
    .sort_values(
        ["f1_macro", "balanced_accuracy", "accuracy"],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(comparacao)

etapa_selecionada = str(comparacao.loc[0, "etapa"])
PESOS_COMBINACAO = comparacao.loc[0, "pesos"]
modelo_final = modelos[etapa_selecionada]
modelo_final.save(MODELO_FINAL)

comparacao.to_csv(
    RESULTADOS / "comparacao_etapas_combinacoes.csv",
    index=False,
)

print("Etapa selecionada:", etapa_selecionada)
print("Pesos:", PESOS_COMBINACAO)

## 15. Ajustar limiares na validação

In [ ]:
saidas_val = modelo_final.predict(
    X_validacao,
    batch_size=256,
    verbose=0,
)
scores_val = combinar_saidas(
    saidas_val,
    PESOS_COMBINACAO,
)
real_val = y_validacao["evento"]

limiares_evento = {"ambiente_normal": 0.0}
linhas_limiares = []

for indice, classe in enumerate(CLASSES):
    if classe == "ambiente_normal":
        continue

    real_binario = (real_val == indice).astype(int)
    candidatos = []

    for limiar in np.arange(0.18, 0.81, 0.01):
        previsto_binario = (
            scores_val[:, indice] >= limiar
        ).astype(int)

        candidatos.append({
            "limiar": float(limiar),
            "precision": float(
                precision_score(
                    real_binario,
                    previsto_binario,
                    zero_division=0,
                )
            ),
            "recall": float(
                recall_score(
                    real_binario,
                    previsto_binario,
                    zero_division=0,
                )
            ),
            "f1": float(
                f1_score(
                    real_binario,
                    previsto_binario,
                    zero_division=0,
                )
            ),
        })

    precisao_minima = 0.68 if classe == "impacto" else 0.72
    aceitaveis = [
        item
        for item in candidatos
        if item["precision"] >= precisao_minima
    ]

    melhor = max(
        aceitaveis or candidatos,
        key=lambda item: (item["f1"], item["recall"]),
    )

    limiares_evento[classe] = round(
        melhor["limiar"],
        2,
    )
    linhas_limiares.append({
        "classe": classe,
        **melhor,
    })

display(pd.DataFrame(linhas_limiares))

(MODELOS / "limiares.json").write_text(
    json.dumps(
        {
            "limiares_evento": limiares_evento,
            "minimo_consecutivas": 2,
            "pesos_combinacao": PESOS_COMBINACAO,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


def aplicar_limiares(scores):
    resultado = []

    for vetor in scores:
        indice = int(np.argmax(vetor))
        classe = CLASSES[indice]

        if classe == "ambiente_normal":
            resultado.append(indice)
        elif vetor[indice] >= limiares_evento[classe]:
            resultado.append(indice)
        else:
            resultado.append(ID["ambiente_normal"])

    return np.asarray(resultado, dtype=np.int32)

## 16. Avaliação oficial no teste limpo

In [ ]:
saidas_teste = modelo_final.predict(
    X_teste,
    batch_size=256,
    verbose=1,
)
scores_teste = combinar_saidas(
    saidas_teste,
    PESOS_COMBINACAO,
)
pred_evento = aplicar_limiares(scores_teste)
real_evento = y_teste["evento"]

accuracy = accuracy_score(real_evento, pred_evento)
balanced = balanced_accuracy_score(real_evento, pred_evento)
f1_macro = f1_score(
    real_evento,
    pred_evento,
    average="macro",
    zero_division=0,
)

relatorio = pd.DataFrame(
    classification_report(
        real_evento,
        pred_evento,
        labels=np.arange(len(CLASSES)),
        target_names=CLASSES,
        output_dict=True,
        digits=4,
        zero_division=0,
    )
).T

display(relatorio)

f1_minimo = float(
    relatorio.loc[CLASSES, "f1-score"].min()
)

atingiu_meta = bool(
    accuracy >= CFG.meta_accuracy
    and f1_macro >= CFG.meta_f1_macro
    and f1_minimo >= CFG.meta_f1_classe
)

resumo = {
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced),
    "f1_macro": float(f1_macro),
    "menor_f1_classe": float(f1_minimo),
    "atingiu_meta": atingiu_meta,
    "etapa_selecionada": etapa_selecionada,
    "pesos_combinacao": PESOS_COMBINACAO,
    "auditoria_split": auditoria_split,
    "metas": {
        "accuracy": CFG.meta_accuracy,
        "f1_macro": CFG.meta_f1_macro,
        "f1_classe": CFG.meta_f1_classe,
    },
}

print(json.dumps(resumo, ensure_ascii=False, indent=2))

relatorio.to_csv(
    RESULTADOS / "metricas_por_classe.csv"
)
(RESULTADOS / "resumo_metricas.json").write_text(
    json.dumps(resumo, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

matriz = confusion_matrix(
    real_evento,
    pred_evento,
    labels=np.arange(len(CLASSES)),
)

plt.figure(figsize=(10, 8))
plt.imshow(matriz, cmap="Blues")
plt.colorbar()
plt.title("Matriz de confusão — AED v8")
plt.xticks(
    range(len(CLASSES)),
    CLASSES,
    rotation=45,
    ha="right",
)
plt.yticks(range(len(CLASSES)), CLASSES)

for linha in range(len(CLASSES)):
    for coluna in range(len(CLASSES)):
        plt.text(
            coluna,
            linha,
            str(matriz[linha, coluna]),
            ha="center",
            va="center",
        )

plt.xlabel("Previsto")
plt.ylabel("Real")
plt.tight_layout()
plt.savefig(
    RESULTADOS / "matriz_confusao.png",
    dpi=180,
)
plt.show()

## 17. Avaliação por gravação com top-3

In [ ]:
tabela_teste_resultado = tabela_teste.copy()

for indice, classe in enumerate(CLASSES):
    tabela_teste_resultado[
        f"prob_{classe}"
    ] = scores_teste[:, indice]

linhas = []

for grupo, parte in tabela_teste_resultado.groupby("grupo"):
    vetor = []

    for classe in CLASSES:
        valores = np.sort(
            parte[f"prob_{classe}"].to_numpy()
        )[::-1]
        k = min(3, len(valores))
        vetor.append(float(valores[:k].mean()))

    vetor = np.asarray(vetor)
    previsto = int(
        aplicar_limiares(vetor[None, :])[0]
    )
    real = int(
        parte["classe_id"].mode().iloc[0]
    )

    linhas.append({
        "grupo": grupo,
        "real": real,
        "previsto": previsto,
        "real_nome": CLASSES[real],
        "previsto_nome": CLASSES[previsto],
        "confianca": float(vetor[previsto]),
    })

gravacoes = pd.DataFrame(linhas)

metricas_gravacao = {
    "accuracy": float(
        accuracy_score(
            gravacoes["real"],
            gravacoes["previsto"],
        )
    ),
    "balanced_accuracy": float(
        balanced_accuracy_score(
            gravacoes["real"],
            gravacoes["previsto"],
        )
    ),
    "f1_macro": float(
        f1_score(
            gravacoes["real"],
            gravacoes["previsto"],
            average="macro",
            zero_division=0,
        )
    ),
}

print(json.dumps(metricas_gravacao, indent=2))

gravacoes.to_csv(
    RESULTADOS / "predicoes_por_gravacao.csv",
    index=False,
)

## 18. Exportação TFLite e paridade

In [ ]:
# Uma única saída fixa simplifica a inferência embarcada.
saidas_modelo = modelo_final(modelo_final.input)
saida_unificada = tf.keras.layers.Concatenate(
    name="saida_unificada"
)([
    saidas_modelo["evento"],
    saidas_modelo["familia"],
    saidas_modelo["vocal"],
    saidas_modelo["transitorio"],
    saidas_modelo["perigo"],
])

modelo_exportacao = tf.keras.Model(
    modelo_final.input,
    saida_unificada,
    name="aed_v8_exportacao",
)

TFLITE_FLOAT32 = (
    MODELOS / "aed_v8_classificador_float32.tflite"
)
TFLITE_FLOAT16 = (
    MODELOS / "aed_v8_classificador_float16.tflite"
)

conversor = tf.lite.TFLiteConverter.from_keras_model(
    modelo_exportacao
)
TFLITE_FLOAT32.write_bytes(conversor.convert())

conversor = tf.lite.TFLiteConverter.from_keras_model(
    modelo_exportacao
)
conversor.optimizations = [tf.lite.Optimize.DEFAULT]
conversor.target_spec.supported_types = [tf.float16]
TFLITE_FLOAT16.write_bytes(conversor.convert())


def prever_tflite(caminho, dados):
    interpretador = tf.lite.Interpreter(
        model_path=str(caminho)
    )
    interpretador.allocate_tensors()
    entrada = interpretador.get_input_details()[0]
    saida = interpretador.get_output_details()[0]
    resultados = []

    for vetor in dados:
        interpretador.set_tensor(
            entrada["index"],
            vetor[None, ...].astype(entrada["dtype"]),
        )
        interpretador.invoke()
        resultados.append(
            interpretador.get_tensor(saida["index"])[0]
        )

    return np.asarray(resultados)


rng = np.random.default_rng(CFG.seed)
quantidade = min(250, len(X_teste))
indices = rng.choice(
    len(X_teste),
    size=quantidade,
    replace=False,
)
amostra = X_teste[indices]
keras_saida = modelo_exportacao.predict(
    amostra,
    batch_size=64,
    verbose=0,
)

linhas_paridade = []

for caminho in (TFLITE_FLOAT32, TFLITE_FLOAT16):
    tflite_saida = prever_tflite(caminho, amostra)
    linhas_paridade.append({
        "modelo": caminho.name,
        "concordancia_evento": float(
            np.mean(
                np.argmax(tflite_saida[:, :6], axis=1)
                == np.argmax(keras_saida[:, :6], axis=1)
            )
        ),
        "diferenca_maxima": float(
            np.max(np.abs(tflite_saida - keras_saida))
        ),
    })

paridade = pd.DataFrame(linhas_paridade)
display(paridade)
paridade.to_csv(
    RESULTADOS / "paridade_tflite.csv",
    index=False,
)

if paridade["concordancia_evento"].min() < CFG.meta_paridade:
    raise RuntimeError("Paridade TFLite abaixo da meta.")

## 19. Teste com áudio externo

In [ ]:
enviados = files.upload()

arquivos_audio = [
    nome
    for nome in enviados
    if Path(nome).suffix.lower() in EXTENSOES
]

if len(arquivos_audio) != 1:
    raise RuntimeError(
        "Envie exatamente um arquivo de áudio."
    )

AUDIO_EXTERNO = Path(arquivos_audio[0]).resolve()
display(Audio(str(AUDIO_EXTERNO)))

audio = carregar_audio(AUDIO_EXTERNO)

if len(audio) <= AMOSTRAS:
    inicios = [0]
else:
    passo = int(CFG.sr * CFG.passo)
    inicios = list(
        range(
            0,
            len(audio) - AMOSTRAS + 1,
            passo,
        )
    )

X_audio = np.asarray([
    extrair_vetor(
        audio[inicio:inicio + AMOSTRAS]
    )
    for inicio in inicios
], dtype=np.float32)

X_audio = scaler.transform(X_audio).astype(np.float32)
saidas_audio = modelo_final.predict(
    X_audio,
    batch_size=64,
    verbose=0,
)
scores_audio = combinar_saidas(
    saidas_audio,
    PESOS_COMBINACAO,
)
previsto_audio = aplicar_limiares(scores_audio)

linhas = []

for inicio, indice, vetor in zip(
    inicios,
    previsto_audio,
    scores_audio,
    strict=True,
):
    linhas.append({
        "inicio": inicio / CFG.sr,
        "fim": inicio / CFG.sr + CFG.duracao,
        "classe": CLASSES[indice],
        "confianca": float(vetor[indice]),
        "ativo": bool(
            indice != ID["ambiente_normal"]
        ),
    })

timeline = pd.DataFrame(linhas)

consecutivas = 0
anterior = None
alertas = []

for linha in timeline.itertuples():
    if linha.ativo and linha.classe == anterior:
        consecutivas += 1
    elif linha.ativo:
        consecutivas = 1
    else:
        consecutivas = 0

    anterior = linha.classe if linha.ativo else None
    alertas.append(consecutivas >= 2)

timeline["alerta_temporal"] = alertas

display(
    timeline.sort_values(
        ["alerta_temporal", "confianca"],
        ascending=False,
    ).head(50)
)

timeline.to_csv(
    RESULTADOS / "timeline_audio_externo.csv",
    index=False,
)

## 20. Pacote final

In [ ]:
configuracao_final = {
    "versao": "aed_v8",
    "classes": CLASSES,
    "configuracao": asdict(CFG),
    "yamnet_handle": YAMNET_HANDLE,
    "dimensao_features": int(X_treino.shape[1]),
    "etapa_selecionada": etapa_selecionada,
    "pesos_combinacao": PESOS_COMBINACAO,
    "limiares_evento": limiares_evento,
    "metricas_teste": resumo,
    "metricas_gravacao": metricas_gravacao,
    "auditoria_split": auditoria_split,
    "atingiu_meta": atingiu_meta,
}

(MODELOS / "configuracao_aed_v8.json").write_text(
    json.dumps(
        configuracao_final,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

texto_implantacao = """
AED v8

1. Capturar áudio mono em 16 kHz.
2. Usar janelas de {duracao:.1f} segundos.
3. Executar YAMNet e calcular as mesmas estatísticas.
4. Calcular MFCC, delta, delta-delta e descritores espectrais.
5. Normalizar com normalizacao.npz.
6. Executar aed_v8_classificador_float16.tflite.
7. Separar a saída em evento/família/vocal/transitório/perigo.
8. Aplicar pesos_combinacao e limiares_evento.
9. Exigir duas janelas consecutivas para alerta.
10. O VAD permanece separado e aciona apenas Whisper/NLP.
""".format(duracao=CFG.duracao).strip()

(MODELOS / "README_IMPLANTACAO.txt").write_text(
    texto_implantacao,
    encoding="utf-8",
)

if ENTREGA.exists():
    shutil.rmtree(ENTREGA)

ENTREGA.mkdir(parents=True, exist_ok=True)

for arquivo in MODELOS.glob("*"):
    if arquivo.is_file():
        shutil.copy2(arquivo, ENTREGA / arquivo.name)

for arquivo in RESULTADOS.glob("*"):
    if arquivo.is_file():
        shutil.copy2(arquivo, ENTREGA / arquivo.name)

ZIP_FINAL = shutil.make_archive(
    "/content/aed_v8_entrega",
    "zip",
    root_dir=str(ENTREGA),
)

print("Meta atingida:", atingiu_meta)
print("Modelo:", MODELO_FINAL)
print("ZIP:", ZIP_FINAL)

files.download(ZIP_FINAL)

## Observações

- O FSD50K e os datasets de choro entram somente no treino; o teste limpo do AED v7 não é contaminado.
- O notebook não cria milhares de cópias artificiais. O equilíbrio é feito por amostragem durante o treinamento.
- FSD50K possui rótulos fracos e pode conter eventos sobrepostos. Clipes ambíguos entre duas classes-alvo são descartados.
- MIVIA continua sendo uma boa fonte complementar para grito, vidro e tiro, mas exige obtenção manual no site oficial.
- Preserve o AED v7 como benchmark. O v8 só substitui o v7 quando o teste limpo comprovar melhora.